<a href="https://colab.research.google.com/github/karthikoo7/Machine_Learning-BDA-/blob/main/Hyper_Parameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
df= pd.read_csv('/content/cleaned_76hd.csv')

In [3]:
X = df.drop(columns=['V58'])
Y = df['V58']

In [4]:
from sklearn.model_selection import train_test_split

In [6]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.3,random_state=7,stratify=Y)

X_train.shape,X_test.shape,Y_train.shape,Y_test.shape

((197, 38), (85, 38), (197,), (85,))

In [7]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

In [10]:
pg = {
    'C':[0.1,0.5,1.0],
    'gamma':[0.5,1,5],
}
gscv = GridSearchCV(SVC(kernel='rbf',random_state=7),
                    param_grid=pg,scoring='average_precision',cv=2,verbose=2)

gscv.fit(X_train,Y_train)

Fitting 2 folds for each of 9 candidates, totalling 18 fits
[CV] END ...................................C=0.1, gamma=0.5; total time=   0.1s
[CV] END ...................................C=0.1, gamma=0.5; total time=   0.1s
[CV] END .....................................C=0.1, gamma=1; total time=   0.0s
[CV] END .....................................C=0.1, gamma=1; total time=   0.1s
[CV] END .....................................C=0.1, gamma=5; total time=   0.0s
[CV] END .....................................C=0.1, gamma=5; total time=   0.1s
[CV] END ...................................C=0.5, gamma=0.5; total time=   0.1s
[CV] END ...................................C=0.5, gamma=0.5; total time=   0.0s
[CV] END .....................................C=0.5, gamma=1; total time=   0.0s
[CV] END .....................................C=0.5, gamma=1; total time=   0.0s
[CV] END .....................................C=0.5, gamma=5; total time=   0.0s
[CV] END .....................................C=0

GridSearchCV(cv=2, estimator=SVC(random_state=7),
             param_grid={'C': [0.1, 0.5, 1.0], 'gamma': [0.5, 1, 5]},
             scoring='average_precision', verbose=2)

In [11]:
gscv.best_params_

{'C': 1.0, 'gamma': 0.5}

In [15]:
from sklearn.metrics import classification_report
svm = SVC(C=1,gamma=0.5,kernel='rbf',random_state=7)
svm.fit(X_train,Y_train)
Y_pred = svm.predict(X_test)
print(classification_report(Y_test,Y_pred))

              precision    recall  f1-score   support

           0       0.80      0.80      0.80        46
           1       0.77      0.77      0.77        39

    accuracy                           0.79        85
   macro avg       0.79      0.79      0.79        85
weighted avg       0.79      0.79      0.79        85



Tune:
RandomForest --
XGBoost --
CatBoost --
LightGradientBoost --

In [18]:
# Random Forest --> max_features, n_estimators
from sklearn.ensemble import RandomForestClassifier

In [20]:
pg = {
    'n_estimators':[50,100,200],
    'max_samples':[0.7,0.8,0.6],
    'max_features':[20,28,33]
}
gscv = GridSearchCV(RandomForestClassifier(oob_score=True,random_state=7),
                    param_grid=pg,scoring='average_precision',cv=2,verbose=2)

gscv.fit(X_train,Y_train)

Fitting 2 folds for each of 27 candidates, totalling 54 fits
[CV] END ..max_features=20, max_samples=0.7, n_estimators=50; total time=   0.2s
[CV] END ..max_features=20, max_samples=0.7, n_estimators=50; total time=   0.3s
[CV] END .max_features=20, max_samples=0.7, n_estimators=100; total time=   0.5s
[CV] END .max_features=20, max_samples=0.7, n_estimators=100; total time=   0.4s
[CV] END .max_features=20, max_samples=0.7, n_estimators=200; total time=   0.9s
[CV] END .max_features=20, max_samples=0.7, n_estimators=200; total time=   0.9s
[CV] END ..max_features=20, max_samples=0.8, n_estimators=50; total time=   0.2s
[CV] END ..max_features=20, max_samples=0.8, n_estimators=50; total time=   0.2s
[CV] END .max_features=20, max_samples=0.8, n_estimators=100; total time=   0.5s
[CV] END .max_features=20, max_samples=0.8, n_estimators=100; total time=   0.4s
[CV] END .max_features=20, max_samples=0.8, n_estimators=200; total time=   0.8s
[CV] END .max_features=20, max_samples=0.8, n_es

GridSearchCV(cv=2,
             estimator=RandomForestClassifier(oob_score=True, random_state=7),
             param_grid={'max_features': [20, 28, 33],
                         'max_samples': [0.7, 0.8, 0.6],
                         'n_estimators': [50, 100, 200]},
             scoring='average_precision', verbose=2)

In [23]:
gscv.best_params_


{'max_features': 20, 'max_samples': 0.7, 'n_estimators': 200}

In [22]:
gscv.best_score_

np.float64(0.8816617941638173)

In [24]:
rf = RandomForestClassifier(max_features=20,max_samples=0.7,n_estimators=200,oob_score=True,random_state=7)
rf.fit(X_train,Y_train)
Y_pred = rf.predict(X_test)
print(classification_report(Y_test,Y_pred))

              precision    recall  f1-score   support

           0       0.79      0.83      0.81        46
           1       0.78      0.74      0.76        39

    accuracy                           0.79        85
   macro avg       0.79      0.78      0.79        85
weighted avg       0.79      0.79      0.79        85



# XGBoost

In [25]:
from xgboost import XGBClassifier

In [45]:
pg = {
    'n_estimators':[50,100,200,300,400,500,1000],
    'learning_rate':[0.1,0.2,0.3,0.5,0.7,0.9],
    'max_depth':[1,2,3,4,5,6,7]
    }
gscv = GridSearchCV(XGBClassifier(subsample=0.8,colsample_bytree=0.8,random_state=7),
                    param_grid=pg,scoring='average_precision',cv=2,verbose=2)

gscv.fit(X_train,Y_train)

Fitting 2 folds for each of 294 candidates, totalling 588 fits
[CV] END ....learning_rate=0.1, max_depth=1, n_estimators=50; total time=   0.3s
[CV] END ....learning_rate=0.1, max_depth=1, n_estimators=50; total time=   1.0s
[CV] END ...learning_rate=0.1, max_depth=1, n_estimators=100; total time=   1.0s
[CV] END ...learning_rate=0.1, max_depth=1, n_estimators=100; total time=   0.4s
[CV] END ...learning_rate=0.1, max_depth=1, n_estimators=200; total time=   0.6s
[CV] END ...learning_rate=0.1, max_depth=1, n_estimators=200; total time=   0.1s
[CV] END ...learning_rate=0.1, max_depth=1, n_estimators=300; total time=   0.1s
[CV] END ...learning_rate=0.1, max_depth=1, n_estimators=300; total time=   0.2s
[CV] END ...learning_rate=0.1, max_depth=1, n_estimators=400; total time=   0.8s
[CV] END ...learning_rate=0.1, max_depth=1, n_estimators=400; total time=   0.3s
[CV] END ...learning_rate=0.1, max_depth=1, n_estimators=500; total time=   0.4s
[CV] END ...learning_rate=0.1, max_depth=1, n_

GridSearchCV(cv=2,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=0.8, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=None...
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             param_grid={'learning_rate': [0.1, 0.2, 0.3, 0.5, 0.7, 0.9],
                         'max_depth': [1, 2, 3, 4, 5, 6, 7],
                         'n_estimators': [50, 100, 200, 300, 400, 500, 1000]},
             scoring='average_precision', verbose=2)

In [46]:
gscv.best_params_


{'learning_rate': 0.1, 'max_depth': 1, 'n_estimators': 50}

In [47]:
gscv.best_score_

np.float64(0.8870856828805861)

In [48]:
xgb = XGBClassifier(subsample=0.8,colsample_bytree=0.8,random_state=7,learning_rate=0.1,max_depth=1,n_estimators=50)
xgb.fit(X_train,Y_train)
Y_pred = xgb.predict(X_test)
print(classification_report(Y_test,Y_pred))


              precision    recall  f1-score   support

           0       0.79      0.89      0.84        46
           1       0.85      0.72      0.78        39

    accuracy                           0.81        85
   macro avg       0.82      0.80      0.81        85
weighted avg       0.82      0.81      0.81        85



CatBoost

In [43]:

!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 5.9 MB/s eta 0:00:00


In [44]:
from catboost import CatBoostClassifier

In [49]:

pg = {
    'iterations':[50,100,150,200,250,300],
    'learning_rate':[0.1,0.2,0.5,0.7,0.9],
    'depth':[1,2,3,4,5,6],
    }

gscv = GridSearchCV(CatBoostClassifier(cat_features=None,loss_function='MultiClass',random_state=7),param_grid=pg,scoring='average_precision',cv=2)

gscv.fit(X_train,Y_train)

Streaming output truncated to the last 5000 lines.
50:	learn: 0.0820232	total: 128ms	remaining: 500ms
51:	learn: 0.0799726	total: 130ms	remaining: 496ms
52:	learn: 0.0787161	total: 132ms	remaining: 492ms
53:	learn: 0.0762853	total: 135ms	remaining: 488ms
54:	learn: 0.0743846	total: 137ms	remaining: 484ms
55:	learn: 0.0715643	total: 139ms	remaining: 480ms
56:	learn: 0.0697715	total: 141ms	remaining: 477ms
57:	learn: 0.0684717	total: 143ms	remaining: 473ms
58:	learn: 0.0670904	total: 145ms	remaining: 469ms
59:	learn: 0.0657732	total: 147ms	remaining: 465ms
60:	learn: 0.0644432	total: 149ms	remaining: 462ms
61:	learn: 0.0631678	total: 151ms	remaining: 458ms
62:	learn: 0.0621267	total: 153ms	remaining: 455ms
63:	learn: 0.0609434	total: 155ms	remaining: 451ms
64:	learn: 0.0599764	total: 157ms	remaining: 448ms
65:	learn: 0.0586035	total: 159ms	remaining: 445ms
66:	learn: 0.0575264	total: 161ms	remaining: 441ms
67:	learn: 0.0562266	total: 164ms	remaining: 438ms
68:	learn: 0.0550536	total: 166

GridSearchCV(cv=2,
             estimator=CatBoostClassifier(loss_function='MultiClass', random_state=7),
             param_grid={'depth': [1, 2, 3, 4, 5, 6],
                         'iterations': [50, 100, 150, 200, 250, 300],
                         'learning_rate': [0.1, 0.2, 0.5, 0.7, 0.9]},
             scoring='average_precision')

In [50]:
gscv.best_params_

{'depth': 1, 'iterations': 50, 'learning_rate': 0.1}

In [51]:
gscv.best_score_

np.float64(0.9039838816340755)

In [53]:
cbc = CatBoostClassifier(depth= 1, iterations= 50, learning_rate= 0.1,cat_features=None,loss_function='MultiClass',random_state=7)
cbc.fit(X_train,Y_train)
Y_pred = cbc.predict(X_test)

print(classification_report(Y_test,Y_pred))

0:	learn: 0.6739792	total: 489us	remaining: 24ms
1:	learn: 0.6574485	total: 895us	remaining: 21.5ms
2:	learn: 0.6428753	total: 1.28ms	remaining: 20ms
3:	learn: 0.6288467	total: 1.63ms	remaining: 18.8ms
4:	learn: 0.6063624	total: 1.97ms	remaining: 17.7ms
5:	learn: 0.5926956	total: 2.3ms	remaining: 16.9ms
6:	learn: 0.5746539	total: 2.64ms	remaining: 16.2ms
7:	learn: 0.5638141	total: 3ms	remaining: 15.8ms
8:	learn: 0.5500981	total: 3.38ms	remaining: 15.4ms
9:	learn: 0.5360740	total: 3.76ms	remaining: 15ms
10:	learn: 0.5244007	total: 4.1ms	remaining: 14.5ms
11:	learn: 0.5167623	total: 4.44ms	remaining: 14.1ms
12:	learn: 0.5156582	total: 4.79ms	remaining: 13.6ms
13:	learn: 0.5111239	total: 5.13ms	remaining: 13.2ms
14:	learn: 0.5033642	total: 5.54ms	remaining: 12.9ms
15:	learn: 0.4946943	total: 5.88ms	remaining: 12.5ms
16:	learn: 0.4883702	total: 6.23ms	remaining: 12.1ms
17:	learn: 0.4792153	total: 6.57ms	remaining: 11.7ms
18:	learn: 0.4716156	total: 6.89ms	remaining: 11.2ms
19:	learn: 0.464

LGBoost

In [54]:
import lightgbm as lgb

In [55]:
pg = {
    'n_estimators':[5000,10000,2000,3000,4000,1000],
    'learning_rate':[0.1,0.2,0.3,0.5,0.7,0.9],
    'max_depth':[1,2,3,4,5,6,7]
    }
gscv = GridSearchCV(lgb.LGBMClassifier(subsample=0.8,colsample_bytree=0.8,random_state=7),
                    param_grid=pg,scoring='average_precision',cv=2,verbose=2)

gscv.fit(X_train,Y_train)

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

GridSearchCV(cv=2,
             estimator=LGBMClassifier(colsample_bytree=0.8, random_state=7,
                                      subsample=0.8),
             param_grid={'learning_rate': [0.1, 0.2, 0.3, 0.5, 0.7, 0.9],
                         'max_depth': [1, 2, 3, 4, 5, 6, 7],
                         'n_estimators': [5000, 10000, 2000, 3000, 4000, 1000]},
             scoring='average_precision', verbose=2)

In [56]:
gscv.best_params_

{'learning_rate': 0.1, 'max_depth': 1, 'n_estimators': 1000}

In [57]:
gscv.best_score_

np.float64(0.8616237861080285)

In [59]:

lgbm = lgb.LGBMClassifier(max_depth=1,learning_rate=0.1,n_estimators=1000,subsample=0.8,colsample_bytree=0.8,random_state=42)
lgbm.fit(X_train,Y_train)
Y_pred = lgbm.predict(X_test)
print(classification_report(Y_test,Y_pred))

[LightGBM] [Info] Number of positive: 89, number of negative: 108
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000126 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 572
[LightGBM] [Info] Number of data points in the train set: 197, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.451777 -> initscore=-0.193495
[LightGBM] [Info] Start training from score -0.193495
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
